In [2]:
"""
IMPROVED ANOMALOUS TRANSACTION DETECTION PIPELINE
Fixes: contamination rate, feature engineering, type-stratified models,
       driver memory, and scoring threshold tuning.
"""

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, StructType, StructField, StringType, LongType
)
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml import Pipeline

import numpy as np
import pandas as pd
import mlflow, mlflow.sklearn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score,
    precision_recall_curve
)

spark = (
    SparkSession.builder
    .appName("ImprovedAnomalyPipeline")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

26/04/06 15:44:38 WARN Utils: Your hostname, Jonathans-MacBook-Pro-16.local resolves to a loopback address: 127.0.0.1; using 192.168.100.190 instead (on interface en0)
26/04/06 15:44:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 15:44:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/06 15:44:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Ingestion

In [3]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/synthetic_mobile_money_transaction_dataset.csv")
)

from pyspark.sql.types import DoubleType, IntegerType
df = (
    df_raw
    .withColumn("step",            F.col("step").cast(IntegerType()))
    .withColumn("amount",          F.col("amount").cast(DoubleType()))
    .withColumn("oldBalInitiator", F.col("oldBalInitiator").cast(DoubleType()))
    .withColumn("newBalInitiator", F.col("newBalInitiator").cast(DoubleType()))
    .withColumn("oldBalRecipient", F.col("oldBalRecipient").cast(DoubleType()))
    .withColumn("newBalRecipient", F.col("newBalRecipient").cast(DoubleType()))
    .withColumn("isFraud",         F.col("isFraud").cast(IntegerType()))
)

critical = ["step", "transactionType", "amount",
            "oldBalInitiator", "newBalInitiator",
            "oldBalRecipient", "newBalRecipient"]
df = df.dropna(subset=critical).filter(F.col("amount") > 0)

total_rows  = df.count()
fraud_rows  = df.filter(F.col("isFraud") == 1).count()
true_fraud_rate = fraud_rows / total_rows
print(f"Total rows      : {total_rows:,}")
print(f"Fraud rows      : {fraud_rows:,}  ({true_fraud_rate*100:.3f}%)")

[Stage 5:===>                                                     (1 + 15) / 16]

Total rows      : 1,720,181
Fraud rows      : 175,518  (10.203%)


## Fraud-Specific Feature Engineering

In [4]:
# remove `step` (noise), add fraud-discriminative signals,
# use richer balance logic, add temporal burst features.

w_init     = Window.partitionBy("initiator").orderBy("step")
w_init_all = Window.partitionBy("initiator")

df = (
    df
    # Core balance signals 
    .withColumn("balance_change_initiator",
                F.col("newBalInitiator") - F.col("oldBalInitiator"))
    .withColumn("balance_change_recipient",
                F.col("newBalRecipient") - F.col("oldBalRecipient"))

    # FIX: amount as fraction of pre-tx balance 
    # Fraudsters typically drain the entire account → ratio near 1.0
    .withColumn("amount_to_initiator_bal_ratio",
                F.when(F.col("oldBalInitiator") > 0,
                       F.col("amount") / F.col("oldBalInitiator"))
                 .otherwise(F.lit(0.0)))

    # post-transaction balance as fraction of amount (how much is left?)
    .withColumn("residual_balance_ratio",
                F.when(F.col("amount") > 0,
                       F.col("newBalInitiator") / F.col("amount"))
                 .otherwise(F.lit(0.0)))

    # Balance discrepancy (accounting error = strong fraud signal)
    .withColumn("balance_discrepancy_initiator",
                F.abs((F.col("newBalInitiator") - F.col("oldBalInitiator"))
                      + F.col("amount")))
    .withColumn("balance_discrepancy_recipient",
                F.abs((F.col("newBalRecipient") - F.col("oldBalRecipient"))
                      - F.col("amount")))

    # hard flag — account fully drained 
    .withColumn("initiator_account_emptied",
                F.when((F.col("oldBalInitiator") > 0) &
                       (F.col("newBalInitiator") <= 0),
                       F.lit(1.0)).otherwise(F.lit(0.0)))

    # recipient had zero balance before receiving (new/mule acct)
    .withColumn("recipient_was_empty",
                F.when(F.col("oldBalRecipient") == 0,
                       F.lit(1.0)).otherwise(F.lit(0.0)))

    # round-number amount flag (fraudsters often pick exact values)
    .withColumn("is_round_amount",
                F.when(F.col("amount") % 1000 == 0,
                       F.lit(1.0)).otherwise(F.lit(0.0)))

    # Log-scale amount
    .withColumn("log_amount", F.log1p(F.col("amount")))

    # personal velocity — how this tx compares to the account mean
    .withColumn("personal_avg_amount",
                F.avg("amount").over(w_init_all))
    .withColumn("amount_vs_personal_avg",
                F.when(F.col("personal_avg_amount") > 0,
                       F.col("amount") / F.col("personal_avg_amount"))
                 .otherwise(F.lit(1.0)))

    # step gap (burst = very short gap from previous transaction)
    .withColumn("prev_step",
                F.lag("step", 1).over(w_init))
    .withColumn("step_gap",
                (F.col("step") - F.col("prev_step")))
    .fillna({"step_gap": 0.0, "personal_avg_amount": 0.0,
             "amount_vs_personal_avg": 1.0})
)

df.cache()
print("Feature engineering complete.")

Feature engineering complete.


## Type-Stratified Models

In [5]:
# fraud in this datasets occurs almost exclusively in TRANSFER
# and CASH_OUT. Training one model across all types dilutes the
# signal from normal PAYMENT / CASH_IN behaviour.
#
# Strategy:
#       • High-risk types  (TRANSFER, CASH_OUT) → dedicated Isolation Forest
#       • Low-risk types   (everything else)    → lightweight statistical model

HIGH_RISK_TYPES = ["TRANSFER", "CASH_OUT"]

df_highrisk = df.filter(F.col("transactionType").isin(HIGH_RISK_TYPES))
df_lowrisk  = df.filter(~F.col("transactionType").isin(HIGH_RISK_TYPES))

print(f"High-risk rows : {df_highrisk.count():,}")
print(f"Low-risk rows  : {df_lowrisk.count():,}")

# Feature list for high-risk model (no transactionTypeIndex needed)
HIGHRISK_FEATURES = [
    "log_amount",
    "amount_to_initiator_bal_ratio",
    "residual_balance_ratio",
    "balance_discrepancy_initiator",
    "balance_discrepancy_recipient",
    "initiator_account_emptied",
    "recipient_was_empty",
    "is_round_amount",
    "amount_vs_personal_avg",
    "step_gap",
    "balance_change_initiator",
    "balance_change_recipient",
]

assembler_hr = VectorAssembler(
    inputCols=HIGHRISK_FEATURES, outputCol="raw_features", handleInvalid="skip"
)
scaler_hr = StandardScaler(
    inputCol="raw_features", outputCol="features",
    withMean=True, withStd=True
)
prep_hr = Pipeline(stages=[assembler_hr, scaler_hr])
prep_model_hr = prep_hr.fit(df_highrisk)
df_hr_scaled  = prep_model_hr.transform(df_highrisk)
df_hr_scaled.cache()

High-risk rows : 569,328
Low-risk rows  : 1,150,853


26/04/06 15:45:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[step: int, transactionType: string, amount: double, initiator: bigint, oldBalInitiator: double, newBalInitiator: double, recipient: string, oldBalRecipient: double, newBalRecipient: double, isFraud: int, balance_change_initiator: double, balance_change_recipient: double, amount_to_initiator_bal_ratio: double, residual_balance_ratio: double, balance_discrepancy_initiator: double, balance_discrepancy_recipient: double, initiator_account_emptied: double, recipient_was_empty: double, is_round_amount: double, log_amount: double, personal_avg_amount: double, amount_vs_personal_avg: double, prev_step: int, step_gap: int, raw_features: vector, features: vector]

## Calibrate Contamination Rate

In [6]:
#   Use the observed fraud rate as a ceiling, with a small buffer.
#   The model flags statistical outliers; not all outliers are fraud,
#   so we allow contamination = 2–3× the fraud rate as a starting point.

hr_rows     = df_hr_scaled.count()
hr_fraud    = df_hr_scaled.filter(F.col("isFraud") == 1).count()
hr_fraud_rate = hr_fraud / hr_rows

# Allow 3× the actual fraud rate — captures fraud with a small false-alarm buffer
CONTAMINATION = min(round(hr_fraud_rate * 3, 4), 0.05)

print(f"\nHigh-risk subset fraud rate : {hr_fraud_rate*100:.3f}%")
print(f"Calibrated contamination    : {CONTAMINATION*100:.3f}%")


High-risk subset fraud rate : 30.829%
Calibrated contamination    : 5.000%


## Training

In [8]:
print("\nTraining Isolation Forest via mapInPandas …")

# Step 5a: drop VectorUDT columns — Arrow cannot cross this boundary 
VECTOR_COLS = ["raw_features", "features"]   # added by VectorAssembler / Scaler
 
df_hr_arrow = df_hr_scaled.drop(*VECTOR_COLS)

df_hr_arrow = df_hr_arrow.repartition(200)
 
#  build output schema from the Arrow-safe frame 
#  Add only the two new scalar output columns; no VectorUDT in sight.
output_schema = (
    df_hr_arrow.schema
    .add(StructField("anomaly_score", DoubleType(),  True))
    .add(StructField("is_anomaly",    IntegerType(), True))
)
 
feat_bc = spark.sparkContext.broadcast(HIGHRISK_FEATURES)
 
def train_and_score_partition(iterator):
    """
    Train IsolationForest on each partition's scalar features; score in place.
    Receives and yields plain pandas DataFrames — no Spark vectors involved.
    """
    feats = feat_bc.value
    for pdf in iterator:
        if len(pdf) < 20:                        # skip undersized partitions
            pdf["anomaly_score"] = 0.0
            pdf["is_anomaly"]    = 0
            yield pdf
            continue
 
        X = pdf[feats].fillna(0.0).values        # numpy array from scalar cols
 
        clf = IsolationForest(
            n_estimators  = 200,
            contamination = CONTAMINATION,
            max_samples   = min(512, len(X)),    # cap per-tree sample for speed
            max_features  = 0.8,                 # feature subsampling adds robustness
            bootstrap     = False,
            random_state  = 42,
            n_jobs        = -1
        )
        clf.fit(X)
 
        # decision_function: lower value = more anomalous; negate for intuition
        pdf["anomaly_score"] = -clf.decision_function(X).astype(float)
        pdf["is_anomaly"]    = (clf.predict(X) == -1).astype(int)
        yield pdf
 
# run mapInPandas on the Arrow-safe (vector-free) DataFrame
df_hr_scored = (
    df_hr_arrow
    .repartition(50)                             # balance partition sizes
    .mapInPandas(train_and_score_partition, schema=output_schema)
)
df_hr_scored.cache()
 
flag_counts = df_hr_scored.groupBy("is_anomaly").count().orderBy("is_anomaly")
flag_counts.show()


Training Isolation Forest via mapInPandas …


[Stage 42:=====================================================>  (48 + 2) / 50]

+----------+------+
|is_anomaly| count|
+----------+------+
|         0|540832|
|         1| 28496|
+----------+------+



## Low-Risk Transaction Types

In [9]:
lr_stats = df_lowrisk.select(
    F.mean("amount").alias("mu"),
    F.stddev("amount").alias("sigma")
).collect()[0]
 
df_lr_scored = (
    df_lowrisk
    .withColumn(
        "anomaly_score",
        F.abs((F.col("amount") - F.lit(lr_stats["mu"]))
              / F.lit(lr_stats["sigma"] or 1.0))
    )
    .withColumn(
        "is_anomaly",
        F.when(F.col("anomaly_score") > 4.0, F.lit(1)).otherwise(F.lit(0))
    )
)

## Threshold Training on High Risk Scores

In [10]:
print("\nTuning score threshold on high-risk subset …")
 
# Sample for threshold tuning (collect only a manageable slice)
SAMPLE_SIZE = min(200_000, hr_rows)
sample_pdf = (
    df_hr_scored
    .select("anomaly_score", "isFraud")
    .sample(fraction=SAMPLE_SIZE / hr_rows, seed=42)
    .toPandas()
)
 
y_true_sample   = sample_pdf["isFraud"].values
scores_sample   = sample_pdf["anomaly_score"].values
 
# Precision-Recall curve sweep
precisions, recalls, thresholds = precision_recall_curve(
    y_true_sample, scores_sample
)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx  = np.argmax(f1_scores)
BEST_THRESHOLD = thresholds[best_idx]
 
print(f"Best threshold : {BEST_THRESHOLD:.4f}")
print(f"  → Precision  : {precisions[best_idx]:.4f}")
print(f"  → Recall     : {recalls[best_idx]:.4f}")
print(f"  → F1         : {f1_scores[best_idx]:.4f}")
 
# Apply best threshold back to Spark DataFrame
df_hr_scored = df_hr_scored.withColumn(
    "is_anomaly_tuned",
    F.when(F.col("anomaly_score") >= F.lit(BEST_THRESHOLD),
           F.lit(1)).otherwise(F.lit(0))
)
 


Tuning score threshold on high-risk subset …
Best threshold : -0.2548
  → Precision  : 0.3071
  → Recall     : 0.9998
  → F1         : 0.4699


## Evaluation

In [12]:
print("FINAL EVALUATION (isFraud as post-hoc reference only)")
 
eval_pdf = (
    df_hr_scored
    .select("anomaly_score", "is_anomaly", "is_anomaly_tuned", "isFraud")
    .toPandas()
)
 
y_true  = eval_pdf["isFraud"].values
y_default  = eval_pdf["is_anomaly"].values
y_tuned    = eval_pdf["is_anomaly_tuned"].values
scores_all = eval_pdf["anomaly_score"].values
 
def report(label, y_true, y_pred):
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)
    n = y_pred.sum()
    print(f"\n  [{label}]")
    print(f"  Flagged      : {n:,}")
    print(f"  Precision    : {p:.4f}")
    print(f"  Recall       : {r:.4f}")
    print(f"  F1-Score     : {f:.4f}")
 
report("Default threshold (contamination-based)", y_true, y_default)
report("Tuned threshold  (F1-maximised)",         y_true, y_tuned)
 
auprc = average_precision_score(y_true, scores_all)
auroc = roc_auc_score(y_true, scores_all)
print(f"\n  AU-PRC (area under PR curve) : {auprc:.4f}")
print(f"  AU-ROC                       : {auroc:.4f}")
 
# Fraud capture breakdown
flagged = eval_pdf[eval_pdf["is_anomaly_tuned"] == 1]
print(f"\n  Flagged transactions         : {len(flagged):,}")
print(f"  Of flagged, known fraudulent : {flagged['isFraud'].sum():,}")
print(f"  Fraud capture rate           : {flagged['isFraud'].mean()*100:.1f}%")
 
# Score distribution by class
print("\n ANOMALY SCORE DISTRIBUTION BY isFraud LABEL")
print(
    eval_pdf.groupby("isFraud")["anomaly_score"]
    .describe().round(4)
)
 

FINAL EVALUATION (isFraud as post-hoc reference only)

  [Default threshold (contamination-based)]
  Flagged      : 28,496
  Precision    : 0.2574
  Recall       : 0.0418
  F1-Score     : 0.0719

  [Tuned threshold  (F1-maximised)]
  Flagged      : 569,185
  Precision    : 0.3083
  Recall       : 0.9997
  F1-Score     : 0.4713

  AU-PRC (area under PR curve) : 0.3087
  AU-ROC                       : 0.5089

  Flagged transactions         : 569,185
  Of flagged, known fraudulent : 175,471
  Fraud capture rate           : 30.8%

 ANOMALY SCORE DISTRIBUTION BY isFraud LABEL
            count    mean     std     min     25%     50%     75%     max
isFraud                                                                  
0        393810.0 -0.1790  0.0702 -0.2616 -0.2150 -0.2008 -0.1786  0.2797
1        175518.0 -0.1804  0.0624 -0.2613 -0.2145 -0.2000 -0.1755  0.2055


## MLFlow Logging

In [13]:
mlflow.set_experiment("compliance_anomaly_detection_v2")
with mlflow.start_run(run_name="IsolationForest_improved"):
    mlflow.log_params({
        "model_type":        "IsolationForest",
        "n_estimators":      200,
        "contamination":     CONTAMINATION,
        "max_samples":       512,
        "max_features":      0.8,
        "stratified_by_type": True,
        "features":          str(HIGHRISK_FEATURES),
        "n_samples":         int(hr_rows),
        "best_threshold":    round(float(BEST_THRESHOLD), 6),
    })
    mlflow.log_metrics({
        "precision_tuned":      float(precision_score(y_true, y_tuned, zero_division=0)),
        "recall_tuned":         float(recall_score(y_true, y_tuned, zero_division=0)),
        "f1_tuned":             float(f1_score(y_true, y_tuned, zero_division=0)),
        "auprc":                float(auprc),
        "auroc":                float(auroc),
        "n_flagged":            int(y_tuned.sum()),
        "fraud_capture_rate":   float(flagged["isFraud"].mean()),
        "true_fraud_rate_pct":  round(hr_fraud_rate * 100, 4),
        "contamination_set":    CONTAMINATION,
    })
    print("\nMLflow run logged.")

2026/04/06 15:55:29 INFO mlflow.tracking.fluent: Experiment with name 'compliance_anomaly_detection_v2' does not exist. Creating a new experiment.



MLflow run logged.


## Union High and Low Risk Results, and Persist

In [14]:
KEEP_COLS = [
    "step", "transactionType", "amount", "initiator",
    "oldBalInitiator", "newBalInitiator",
    "recipient", "oldBalRecipient", "newBalRecipient",
    "isFraud", "anomaly_score", "is_anomaly"
]
 
# Align schemas before union
df_hr_final = df_hr_scored.select(KEEP_COLS)
 
df_lr_final = (
    df_lr_scored
    .withColumnRenamed("is_anomaly", "is_anomaly_lr")
    .select(
        *[c for c in KEEP_COLS if c not in ("anomaly_score", "is_anomaly")],
        F.col("anomaly_score"),
        F.col("is_anomaly_lr").alias("is_anomaly")
    )
)
 
df_final = df_hr_final.unionByName(df_lr_final)
 
# df_final.write.parquet("/output/anomaly_scored_transactions/", mode="overwrite")
 
print("\n Pipeline complete. Outputs ready.")
print("   df_final columns:", df_final.columns)


 Pipeline complete. Outputs ready.
   df_final columns: ['step', 'transactionType', 'amount', 'initiator', 'oldBalInitiator', 'newBalInitiator', 'recipient', 'oldBalRecipient', 'newBalRecipient', 'isFraud', 'anomaly_score', 'is_anomaly']


In [15]:
# Are fraud transactions actually distinguishable in the engineered features?
# If these distributions overlap heavily, models will struggle to separate them.

feature_check = df_hr_scored.groupBy("isFraud").agg(
    F.mean("amount_to_initiator_bal_ratio").alias("avg_amt_bal_ratio"),
    F.mean("initiator_account_emptied").alias("avg_acct_emptied"),
    F.mean("balance_discrepancy_initiator").alias("avg_bal_discrepancy"),
    F.mean("recipient_was_empty").alias("avg_recip_empty"),
    F.mean("amount_vs_personal_avg").alias("avg_vs_personal"),
    F.mean("log_amount").alias("avg_log_amount"),
).orderBy("isFraud")

feature_check.show(truncate=False)

[Stage 66:=======================================>               (36 + 14) / 50]

+-------+------------------+---------------------+-------------------+-------------------+------------------+------------------+
|isFraud|avg_amt_bal_ratio |avg_acct_emptied     |avg_bal_discrepancy|avg_recip_empty    |avg_vs_personal   |avg_log_amount    |
+-------+------------------+---------------------+-------------------+-------------------+------------------+------------------+
|0      |1.5669881652541349|0.0031207942916634925|1859.5897364465065 |0.3478885757090983 |0.9282472827806638|10.186537343537003|
|1      |1.5183314920737312|0.002968356521838216 |2513.877775612774  |0.36449253068061394|0.7180040632265647|10.030313387327913|
+-------+------------------+---------------------+-------------------+-------------------+------------------+------------------+

